# Programme scores across epithelial clusters

This notebook generates Figure 1d for the run selected by `CAP_RUN_ID`. It requires completed outputs from `03a_epithelial_integration` and `03b_meta_programmes` for the same run.

It combines epithelial Leiden-cluster assignments from stage `03a_epithelial_integration` with the seven programme-class scores from stage `03b_meta_programmes`. Figures and summary tables are written to stage `04_programme_summary` under `$CAP_DATA_ROOT/runs/$CAP_RUN_ID/`.

In [9]:
from pathlib import Path
import json

import h5py
from anndata.io import read_elem
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D

from cap.figures import (
    configure_figure_style,
    save_figure,
    save_umap_figure,
)
from cap.paths import get_paths
from cap.provenance import (
    sha256_file,
    validate_frozen_inputs,
    write_run_provenance,
)

In [10]:
# CAP_DATA_ROOT and CAP_RUN_ID are read from the environment or repository .env.
paths = get_paths()
paths.validate_base_inputs()
validate_frozen_inputs(paths)
project = paths.root
analysis_dir = paths.run_root
configure_figure_style()

In [11]:
# Read the completed meta-programme stage from the selected immutable run.
source_dir = paths.stage_dir("03b_meta_programmes")
paths.require_dir(source_dir, "03b_meta_programmes stage")
adata_file = source_dir / "h5ad" / "epithelial_meta_programmes.h5ad"
scores_file = source_dir / "tables" / "meta_programme_class_scores.csv"
eligibility_file = source_dir / "tables" / "meta_programme_sample_eligibility.csv"
paths.require_file(adata_file, "03b epithelial meta-programme H5AD")
paths.require_file(scores_file, "03b meta-programme class scores")
paths.require_file(eligibility_file, "03b meta-programme sample eligibility")

WindowsPath('D:/CAP_Data/runs/cap_preprocessing_20261120_210737/03b_meta_programmes/tables/meta_programme_sample_eligibility.csv')

In [12]:
notebook_path = paths.root / "notebooks" / "004_Programme_Summary.ipynb"
paths.require_file(notebook_path, "004 notebook source")

input_checksums = {
    "03b_meta_programmes/h5ad/epithelial_meta_programmes.h5ad": sha256_file(adata_file),
    "03b_meta_programmes/tables/meta_programme_class_scores.csv": sha256_file(scores_file),
    "03b_meta_programmes/tables/meta_programme_sample_eligibility.csv": sha256_file(eligibility_file),
    "frozen_inputs.json": sha256_file(paths.manifest_dir / "frozen_inputs.json"),
}

# Single reservation: refuses a non-empty stage and creates nothing else.
output_dir = paths.prepare_stage("04_programme_summary")
fig_dir = output_dir / "figures"
df_dir = output_dir / "tables"
fig_dir.mkdir()
df_dir.mkdir()

(output_dir / "input_checksums.json").write_text(
    json.dumps(input_checksums, indent=2) + "\n", encoding="utf-8"
)
write_run_provenance(paths, output_dir,
                     notebook_path=notebook_path, input_checksums=input_checksums)

FileExistsError: Stage output already exists and is non-empty: D:\CAP_Data\runs\cap_preprocessing_20261120_210737\04_programme_summary. Set a new CAP_RUN_ID rather than overwriting it.

FileExistsError: Stage output already exists and is non-empty: D:\CAP_Data\runs\cap_preprocessing_20261120_210737\04_programme_summary. Set a new CAP_RUN_ID rather than overwriting it.

In [ ]:
# Read cell metadata and the existing class scores.
with h5py.File(adata_file, "r") as handle:
    obs = read_elem(handle["obs"])
scores = pd.read_csv(scores_file, index_col=0)
eligibility = pd.read_csv(eligibility_file)

In [ ]:
programmes = [
    "Cell Cycle", "Stress or Hypoxia", "Mesenchymal",
    "Interferon and MHC II", "Senescence", "Respiration", "Secreted",
]
# Use the existing cluster-summary coverage thresholds from 003b.
min_cluster_cells = 10
min_other_cells = 10
min_donors = 3

In [ ]:
# Confirm that all eligible cells belong to this saved object.
selected_samples = eligibility.loc[eligibility["included"], "sample"]
expected_cells = obs.index[obs["sample"].isin(selected_samples)]
assert scores.index.is_unique and scores.index.equals(expected_cells)
assert np.isfinite(scores[programmes].to_numpy()).all()
cells = obs.loc[scores.index, ["sample", "patient", "condition", "epi_leiden"]].copy()
cells = cells.rename(columns={"epi_leiden": "cluster"})
cells["cluster"] = cells["cluster"].astype(str)
assert cells.notna().all().all() and cells["condition"].ne("N").all()
assert cells.groupby("patient", observed=True)["condition"].nunique().eq(1).all()
print(f"{len(cells):,} cells, {cells['sample'].nunique()} libraries, {cells.patient.nunique()} donors")

## Donor-level summary

Scores are centred on each library's mean across all eligible cells. Library–cluster
means contribute when both the cluster and its remainder contain at least ten cells.
Technical libraries are averaged within donor, followed by equal weighting of donors.
Colour shows the mean centred class score. Dot area shows the fraction of contributing
donors with a positive deviation. Crosses identify fewer than three contributing donors.
All eligible cells contribute, including low-activity and ambiguous programme calls.

In [ ]:
# Centre within each library before calculating cluster means.
centred_scores = scores[programmes] - scores[programmes].groupby(
    cells["sample"], observed=True
).transform("mean")
keys = ["sample", "patient", "condition", "cluster"]
library_scores = centred_scores.join(cells).groupby(keys, observed=True)[programmes].mean()
library_scores["n_cells"] = cells.groupby(keys, observed=True).size()
library_totals = cells.groupby("sample", observed=True).size()
library_scores["n_other_cells"] = (
    library_scores.index.get_level_values("sample").map(library_totals).to_numpy()
    - library_scores["n_cells"].to_numpy()
)
library_scores["eligible"] = (
    library_scores.n_cells.ge(min_cluster_cells)
    & library_scores.n_other_cells.ge(min_other_cells)
)

In [ ]:
# Give technical libraries equal weight within each donor.
contributing = library_scores.loc[library_scores["eligible"]]
donor_scores = contributing.groupby(
    level=["patient", "condition", "cluster"], observed=True
)[programmes].mean()
donor_long = donor_scores.reset_index().melt(
    id_vars=["patient", "condition", "cluster"], value_vars=programmes,
    var_name="programme", value_name="centred_score",
)

In [ ]:
# Retain every cluster in numerical order, including limited donor coverage.
clusters = sorted(obs["epi_leiden"].astype(str).unique(), key=int)
summary = donor_long.groupby(["cluster", "programme"], observed=True).agg(
    mean_centred_score=("centred_score", "mean"),
    fraction_donors_positive=("centred_score", lambda values: values.gt(0).mean()),
    n_donors=("patient", "nunique"),
).reindex(pd.MultiIndex.from_product([clusters, programmes], names=["cluster", "programme"]))
summary["n_donors"] = summary.n_donors.fillna(0).astype(int)
summary["eligible"] = summary.n_donors.ge(min_donors)
summary["n_cells_in_primary_population"] = summary.index.get_level_values("cluster").map(
    cells.groupby("cluster", observed=True).size()
).fillna(0).astype(int)

In [ ]:
plot_data = summary.reset_index()
plot_data["x"] = plot_data.programme.map(dict(zip(programmes, range(len(programmes)))))
plot_data["y"] = plot_data.cluster.map(dict(zip(clusters, range(len(clusters)))))
shown = plot_data.loc[plot_data["eligible"]]
limited = plot_data.loc[~plot_data["eligible"]]
donor_counts = summary.groupby(level="cluster").n_donors.first()
colour_limit = shown.mean_centred_score.abs().max()
assert colour_limit > 0
max_dot_area = 170

In [ ]:
plt.rcParams.update({"pdf.fonttype": 42, "svg.fonttype": "none", "font.size": 10})
fig, ax = plt.subplots(figsize=(6.8, 6.5))
points = ax.scatter(
    shown.x, shown.y, s=max_dot_area * shown.fraction_donors_positive,
    c=shown.mean_centred_score, cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=-colour_limit, vcenter=0, vmax=colour_limit),
    edgecolors="#333333", linewidths=0.35, zorder=3,
)
ax.scatter(limited.x, limited.y, marker="x", s=16, color="#AAAAAA", linewidths=0.7)
ax.set(xticks=range(len(programmes)), yticks=range(len(clusters)),
       yticklabels=[f"{cluster}  ({donor_counts.loc[cluster]})" for cluster in clusters],
       xlim=(-0.6, len(programmes) - 0.4), ylim=(len(clusters) - 0.4, -0.6),
       ylabel="Epithelial cluster (contributing donors)")
ax.set_xticklabels(programmes, rotation=45, ha="right")
ax.set_axisbelow(True)
ax.grid(color="#ECECEC", linewidth=0.5)
ax.spines[["top", "right"]].set_visible(False)
colourbar = fig.colorbar(points, ax=ax, fraction=0.035, pad=0.04, label="Mean centred class score")
colourbar.solids.set_rasterized(False)
size_handles = [Line2D([], [], linestyle="", marker="o", markersize=np.sqrt(max_dot_area * fraction),
                       markerfacecolor="white", markeredgecolor="#333333", label=f"{fraction:.0%}")
                for fraction in [0.25, 0.5, 1.0]]
size_handles += [Line2D([], [], linestyle="", marker="o", markersize=0, label="0% (no dot)"),
                 Line2D([], [], linestyle="", marker="x", color="#AAAAAA", label="<3 donors")]
ax.legend(handles=size_handles, title="Donors with positive deviation", frameon=False,
          loc="lower left", bbox_to_anchor=(-0.03, 1.02), ncol=3, fontsize=9, title_fontsize=10)
save_figure(fig, fig_dir / "programme_cluster_summary", close=False, overwrite=False)
plt.show()

In [ ]:
# Export the complete signed values and the contributing library/donor scores.
summary.to_csv(df_dir / "programme_cluster_summary.csv")
library_scores.to_csv(df_dir / "programme_cluster_library_scores.csv")
donor_scores.to_csv(df_dir / "programme_cluster_donor_scores.csv")

In [ ]:
# Record the exact inputs used for this panel.
input_files = {"epithelial_object": adata_file, "class_scores": scores_file,
               "sample_eligibility": eligibility_file}
input_hashes = {
    name: sha256_file(path)
    for name, path in input_files.items()
}

In [ ]:
record = {
    "input_files": {name: str(path.relative_to(analysis_dir)) for name, path in input_files.items()},
    "input_sha256": input_hashes,
    "n_cells": len(cells), "n_libraries": cells["sample"].nunique(),
    "n_donors": cells.patient.nunique(), "n_clusters": len(clusters),
    "programme_classes": programmes,
    "minimum_cluster_cells_per_library": min_cluster_cells,
    "minimum_other_cells_per_library": min_other_cells,
    "minimum_contributing_donors": min_donors,
    "colour": "mean library-centred class score, averaged across libraries within donor and then donors",
    "area": "fraction of contributing donors with a positive centred score",
    "summary": "descriptive",
}
(df_dir / "programme_cluster_summary.json").write_text(json.dumps(record, indent=2) + "\n")

The seven classes provide a compact view of the existing programme analysis.
The source table retains signed scores for every cluster. A zero positive-donor
fraction produces zero dot area. Donor counts are given beside each cluster.

In [ ]:
import session_info
session_info.show()